# 新增空间/时空验证数据复核
使用conda sparrow。只核对公开CSV与覆盖，不启动拟合；默认工作目录为本notebook所在目录。

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
root = Path.cwd()
assert (root/'quality.json').exists(), 'Run from expert/tn_transfer_validation'
old = pd.read_csv(root.parent/'tn_challenge_plain/data/observations_all.csv')
new = pd.read_csv(root/'evaluation_labels.csv')
coverage = pd.read_csv(root/'station_year_coverage.csv')


In [ ]:
assert not set(old.observation_id) & set(new.observation_id)
assert not set(old.station_key) & set(new.station_key)
assert not set(old.global_reach_id) & set(new.global_reach_id)
assert new.observation_id.is_unique
assert not new.duplicated(['station_key','year','month']).any()
print(len(new), new.station_key.nunique(), 'new rows/stations; no overlap')


In [ ]:
new.groupby('benchmark_group').agg(rows=('tn_mg_l','size'), stations=('station_key','nunique'), median_TN=('tn_mg_l','median'), max_TN=('tn_mg_l','max'))


In [ ]:
stats = new.groupby(['year','station_key']).tn_mg_l.agg(n='size', variance=lambda x: np.var(x))
stats['NSE_eligible']=(stats.n>=8)&(stats.variance>0)
stats.loc[[2023,2024]].groupby('year').agg(rows=('n','sum'), stations=('n','size'), NSE_eligible=('NSE_eligible','sum'))


In [ ]:
pd.read_csv(root/'static_support_audit.csv')


超出原训练裁剪范围不代表新数据错误；它表示对冻结参数映射的外推挑战。先冻结方法，再预测；这些资料是回顾性公开数据，不称前瞻盲测。当前没有专家新模型的性能结果。